In [1]:
import pandas as pd
import ast

BASE = "/ssd1/jueon/wj/detoxicity_model/molecular_feature/stereochemistry"
loose_df = pd.read_csv(f"{BASE}/pairs_stereo_diff_only_loose.csv")
strict_df = pd.read_csv(f"{BASE}/pairs_stereo_diff_only.csv")

def _parse_list(s):
    if pd.isna(s) or s == "[]" or str(s).strip() == "":
        return []
    try:
        out = ast.literal_eval(s) if isinstance(s, str) else s
        return out if isinstance(out, list) else []
    except Exception:
        return []

def print_pair_example(row, title="Pair"):
    """데이터셋, 엔드포인트, SMILES, 각 SMILES의 Stereochemistry 정보를 보기 좋게 출력."""
    print("=" * 80)
    print(f"  {title}")
    print("=" * 80)
    print(f"  dataset_name : {row['dataset_name']}")
    print(f"  endpoint     : {row['endpoint']}")
    print()
    print("  [Toxic SMILES]")
    print(f"    {row['toxic_smiles']}")
    t_chiral = _parse_list(row.get('toxic_chiral_centers', []))
    t_ez = _parse_list(row.get('toxic_ez_bonds', []))
    print(f"    has_chirality: {row.get('toxic_has_chirality', '')}  (chiral centers: {len(t_chiral)})")
    if t_chiral:
        for c in t_chiral:
            if isinstance(c, dict):
                print(f"      - atom_idx {c.get('atom_idx')}: {c.get('config', '')}")
    print(f"    has_ez_bonds: {row.get('toxic_has_ez_bonds', '')}  (E/Z bonds: {len(t_ez)})")
    if t_ez:
        for b in t_ez:
            if isinstance(b, dict):
                print(f"      - {b.get('bond', b)}: {b.get('geometry', '')}")
    print()
    print("  [Nontoxic SMILES]")
    print(f"    {row['nontoxic_smiles']}")
    n_chiral = _parse_list(row.get('nontoxic_chiral_centers', []))
    n_ez = _parse_list(row.get('nontoxic_ez_bonds', []))
    print(f"    has_chirality: {row.get('nontoxic_has_chirality', '')}  (chiral centers: {len(n_chiral)})")
    if n_chiral:
        for c in n_chiral:
            if isinstance(c, dict):
                print(f"      - atom_idx {c.get('atom_idx')}: {c.get('config', '')}")
    print(f"    has_ez_bonds: {row.get('nontoxic_has_ez_bonds', '')}  (E/Z bonds: {len(n_ez)})")
    if n_ez:
        for b in n_ez:
            if isinstance(b, dict):
                print(f"      - {b.get('bond', b)}: {b.get('geometry', '')}")
    if 'stereo_diff_type' in row:
        print()
        print(f"  [Strict] stereo_diff_type: {row.get('stereo_diff_type', '')}")
    if 'stereo_diff_type_loose' in row:
        print(f"  [Loose]  stereo_diff_type_loose: {row.get('stereo_diff_type_loose', '')}")
    print()

## 그냥(strict) vs Loose 차이

- **그냥(strict)**: 둘 다 chiral center가 있고, 둘 다 E/Z bond가 있을 때만 **설정(R/S, E/Z)이 다를 때** stereo 차이로 봄.
- **Loose**: 한쪽만 chiral/EZ 있어도, **개수만 달라도**, R/S·E/Z 개수만 달라도 차이로 봄 (Mol_stereo와 동일).

In [ ]:
# 그냥(strict) 예시 1개 — 둘 다 chiral 있고, 설정이 다른 경우
print_pair_example(strict_df.iloc[0], "Strict 예시 (둘 다 chiral, 설정 다름)")

In [ ]:
# Loose에만 있고 strict에는 없는 pair (한쪽만 chiral 등으로 loose에서만 차이로 잡힌 경우)
key_cols = ["dataset_name", "endpoint", "toxic_smiles", "nontoxic_smiles"]
strict_keys = set(strict_df[key_cols].astype(str).apply(tuple, axis=1))
loose_keys = loose_df[key_cols].astype(str).apply(tuple, axis=1)
loose_only_df = loose_df.loc[~loose_keys.isin(strict_keys)].reset_index(drop=True)
print(f"Strict pair 수: {len(strict_df):,}")
print(f"Loose pair 수: {len(loose_df):,}")
print(f"Loose에만 있는 pair 수: {len(loose_only_df):,}")

In [ ]:
# Loose 전용 예시 1개 — 한쪽만 chiral 있음 (toxic는 stereo 없음, nontoxic만 있음)
if len(loose_only_df) > 0:
    print_pair_example(loose_only_df.iloc[0], "Loose 전용 예시 (한쪽만 chiral 등)")

In [2]:
loose_df = pd.read_csv("/ssd1/jueon/wj/detoxicity_model/molecular_feature/stereochemistry/pairs_stereo_diff_only_loose.csv")
strict_df = pd.read_csv("/ssd1/jueon/wj/detoxicity_model/molecular_feature/stereochemistry/pairs_stereo_diff_only.csv")

In [4]:
loose_df.head(2)

,dataset_name,endpoint,toxic_smiles,nontoxic_smiles,toxic_scaffold_smiles,nontoxic_scaffold_smiles,tanimoto_sim,delta_MW,delta_logP,delta_TPSA,...,toxic_ez_bonds,toxic_has_chirality,toxic_has_ez_bonds,nontoxic_chiral_centers,nontoxic_ez_bonds,nontoxic_has_chirality,nontoxic_has_ez_bonds,chiral_diff_loose,ez_diff_loose,stereo_diff_type_loose
0,sider_train,Blood and lymphatic system disorders,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,CC1CC2C3CCC4=CC(=O)C=CC4(C)[C@@]3(F)C(O)CC2(C)...,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,0.847458,17.966113,1.2465,20.23,...,[],False,False,"[{'atom_idx': 15, 'config': 'R'}, {'atom_idx':...",[],True,False,True,False,chiral_only
1,sider_train,Blood and lymphatic system disorders,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,CCCCC(=O)O[C@]1(C(=O)CO)[C@@H](C)C[C@H]2[C@@H]...,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,0.731343,84.057515,1.7411,6.07,...,[],False,False,"[{'atom_idx': 7, 'config': 'S'}, {'atom_idx': ...",[],True,False,True,False,chiral_only
